## Visualization for paper

In [ ]:
import os

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas
from xgboost import XGBRegressor

### Data loading

In [ ]:
# Load data and model
total_dataset = pandas.read_parquet(
    "./data/total_dataset.parquet", engine="pyarrow"
)

synthetic_dataset = pandas.read_parquet(
    "./data/synthetic_dataset.parquet", engine="pyarrow"
)

trained_xgb_model = XGBRegressor()
trained_xgb_model.load_model("./data/xgboost_model.bin")

In [ ]:
def plot_cv_timeseries(df_synthetic):
    """ """

    country_ground_truth = df_synthetic["load_mw_percentage"].values
    country_predictions = df_synthetic["predictions"].values
    time_index = pandas.to_datetime(df_synthetic["time_utc"])

    # Create a new figure
    plt.figure(figsize=(15, 8))

    # Plot the ground truth
    plt.plot(
        time_index,
        country_ground_truth,
        label="Ground Truth",
        color="lightgreen",
        alpha=0.7,
    )

    # Plot the predictions
    plt.plot(
        time_index,
        country_predictions,
        label="Predictions",
        color="skyblue",
        alpha=0.7,
    )

    # Plot the absolute difference between predictions and ground truth
    # plt.bar(
    #     numpy.arange(0, len(df_synthetic)),
    #     abs(country_predictions - country_ground_truth),
    #     label="Difference",
    #     color="lightcoral",
    #     alpha=0.7,
    # )

    country_code = df_synthetic["region_code"].iloc[0]
    plt.title(f"{country_code}: Actual vs Predicted Hourly Demand")

    plt.ylabel("Normalized Hourly Demand")
    plt.ylim(
        0,
        round(
            max(
                values_syn_ES["load_mw_percentage"].max(),
                values_syn_ES["predictions"].max(),
            )
            * 1.5,
            4,
        ),
    )
    plt.legend(loc="upper right")

    # Format x-axis with datetime ticks
    ax = plt.gca()

    # Set major ticks to show years
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    # Set minor ticks to show months
    # ax.xaxis.set_minor_locator(mdates.MonthLocator())

    # Rotate labels for better readability
    plt.xticks(rotation=45)

    plt.xlabel("Time")

    # Add grid for better readability
    plt.grid(True, linestyle="--", alpha=0.2)

    # Improve layout
    plt.tight_layout()

    # Save the figure
    plt.savefig(
        os.path.join("./data/" + country_code + "_comparison.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

In [ ]:
# Plot the worst performing region CA_AB
plot_cv_timeseries(
    synthetic_dataset[synthetic_dataset["region_code"] == "CA_AB"]
)

In [ ]:
# Plot the best performing region ES
plot_cv_timeseries(synthetic_dataset[synthetic_dataset["region_code"] == "ES"])